In [1]:
import treeswift as ts
import pandas as pd
import sys
import dendropy
from collections import defaultdict

In [3]:
!nw_topology -I -b ../63K.tre | nw_order - > ../63K-reordered.tre
!nw_topology -I -b ../timetrees/2023-04-dated.nw | nw_order - > ../timetrees/2023-04-dated-reordered.nw
!nw_topology -I -b ../timetrees/63K_dated.tre | nw_order - > ../timetrees/63K_dated-reordered.nw

In [4]:
df = pd.read_csv("../substitution_rates/intergenic_region_ratemult_dates_363.csv")
ttime = ts.read_tree_newick("../timetrees/63K_dated-reordered.nw")
tsub = ts.read_tree_newick("../substitution_rates/species_tree-subs.tre")

In [5]:
tage = ttime.furthest_from_root()
lbl_to_nd = ttime.label_to_node(selection="all")

In [6]:
ndp = zip(ttime.traverse_postorder(internal=True), tsub.traverse_postorder(internal=True))
for nd_time, nd_sub in ndp:
    assert(nd_time.get_label() == nd_sub.get_label())

In [7]:
G = 10/10**6

tns = dendropy.TaxonNamespace()
ttime = dendropy.Tree.get(path="../timetrees/63k_dated.tre", schema='newick', taxon_namespace=tns, preserve_underscores=True)
tcuo = dendropy.Tree.get(path="../63K.tre", schema='newick', taxon_namespace=tns, preserve_underscores=True)

In [8]:
root = ttime.seed_node
l1, l2 = (nd.edge_length for nd in ttime.seed_node.child_nodes())

In [9]:
ttime.deroot()
tcuo.deroot()

for edge in tcuo.levelorder_edge_iter():
    if edge.length is None:
        edge.length = 0

length_diffs = dendropy.calculate.treecompare._get_length_diffs(ttime, tcuo)

i=0
nd_ltime = [nd for nd in ttime.postorder_node_iter()]
nd_lcu = [nd for nd in tcuo.postorder_node_iter()]
nd_l = zip(nd_lcu, nd_ltime)
# root = None
lbl_to_popsize = {}
lbl_to_ngen = {}
lbl_to_cu = {}
lbl_to_time = {}
for nd_time in ttime.postorder_node_iter():
    if not nd_time.is_leaf():
        nd_time.label = f"N{i}"
        time, cu = length_diffs[i]
        if time == 0 and cu == 0:
            # root = nd_time
            for nd_child in nd_time.child_nodes():  
                cu += lbl_to_cu[nd_child.label]
                time += lbl_to_time[nd_child.label]
                # print(time)
        g = time / G
        psize = g / cu
        # print(psize)
        lbl_to_time[nd_time.label] = time
        lbl_to_cu[nd_time.label] = cu
        lbl_to_popsize[nd_time.label] = psize
        for nd_child in nd_time.child_nodes():  
            if nd_child.is_leaf():
                lbl_to_popsize[nd_child.taxon.label] = psize
    i += 1
ttime.scale_edges(1/G)

for nd in ttime.postorder_node_iter():
    if nd.is_leaf():
        lbl_to_ngen[nd.taxon.label] = nd.edge_length
    else:
        lbl_to_ngen[nd.label] = nd.edge_length

In [10]:
ndr = [nd for nd in root.child_nodes()][0]
root = ttime.reroot_at_edge(ndr.edge, l2/G, l1/G)
root.label = "root"
lbl_to_popsize[root.label] = lbl_to_popsize[ndr.label]

In [11]:
with open("../main-num_generations.nwk", "w") as dest:
    ttime.write(file=dest, schema="newick")

In [12]:
ann_to_lbls = defaultdict(list)
with open("../annotation.txt", 'r') as f:
    for l in f.readlines():
        lbl, ann = l.strip().split()
        ann_to_lbls[ann].append(lbl)

In [13]:
ttime = ts.read_tree_newick("../main-num_generations.nwk")
lbl_to_nd = ttime.label_to_node(selection="all")
lbl_to_ann = {}
for ann, lbls in ann_to_lbls.items():
    if len(lbls) > 1:
        nd = ttime.mrca(lbls)
        lbl_to_ann[nd.get_label()] = ann
        nd.set_label(ann)
ttime.write_tree_newick("../main-num_generations.nwk")

In [14]:
with open("../num-generations.tsv", 'w') as f:
    f.write(f"LABEL\tNGEN")
    for lbl, ngen in lbl_to_ngen.items():
        lbl = lbl_to_ann.get(lbl, lbl)
        f.write(f"\n{lbl}\t{ngen}")

In [15]:
with open("../population-sizes.tsv", 'w') as f:
    f.write(f"LABEL\tSIZE")
    for lbl, size in lbl_to_popsize.items():
        lbl = lbl_to_ann.get(lbl, lbl)
        f.write(f"\n{lbl}\t{size}")